# SO-101 橙色方块数据集可视化

这个 notebook 展示 [Pi0.5 模型](https://huggingface.co/felixmayor/pi05_so101_orange_cube) 使用的 [LeRobot 训练集](https://huggingface.co/datasets/felixmayor/orange_cube_merged)：每条 episode 的任务、FPV／俯视视频、6 维关节状态和动作。这里展示的是**录制的示教数据**，不是模型推理结果。

数据集固定在 revision `c021b3c22a3de4e70e81010e54fb250a5dde348b`，共 154 条 episode、68,468 帧、30 FPS。直接运行所有单元格；修改 `EPISODE_INDEX` 或在最后的下拉框中选择轨迹。需要设置路径时可在启动 JupyterLab 前导出 `ORANGE_CUBE_DATASET_ROOT`。

启动命令：
```bash
source /mnt/ceph-zjk1-csp/mm-base-plt2/nrwu/venvs/orange-cube-viz/bin/activate
jupyter lab --no-browser --allow-root \
  --ip="${__HOST_IP__:?__HOST_IP__ is unset}" \
  --port=8080 \
  --ServerApp.root_dir=/mnt/ceph-zjk1-csp/mm-base-plt2/nrwu/work/carrot
```


In [ ]:
import json
import os
import subprocess
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pyarrow.parquet as pq
from IPython.display import Markdown, Video, display


def find_dataset_root():
    override = os.environ.get("ORANGE_CUBE_DATASET_ROOT")
    if override:
        return Path(override).expanduser().resolve()
    for path in (Path.cwd(), *Path.cwd().parents):
        if path.name == "carrot" and path.parent.name == "work":
            return path.parent.parent / "hf-hub/felixmayor/orange_cube_merged"
    assert override, "Set ORANGE_CUBE_DATASET_ROOT to the downloaded dataset directory"


DATASET_ROOT = find_dataset_root()
INFO_PATH = DATASET_ROOT / "meta/info.json"
EPISODES_PATH = DATASET_ROOT / "meta/episodes/chunk-000/file-000.parquet"
assert INFO_PATH.is_file() and EPISODES_PATH.is_file(), (
    f"Dataset metadata is missing in {DATASET_ROOT}"
)

info = json.loads(INFO_PATH.read_text())
episode_columns = [
    "episode_index", "tasks", "length", "dataset_from_index", "data/chunk_index", "data/file_index",
    "videos/observation.images.fpv/chunk_index",
    "videos/observation.images.fpv/file_index",
    "videos/observation.images.fpv/from_timestamp",
    "videos/observation.images.top/chunk_index",
    "videos/observation.images.top/file_index",
    "videos/observation.images.top/from_timestamp",
]
episodes = {
    row["episode_index"]: row
    for row in pq.read_table(EPISODES_PATH, columns=episode_columns).to_pylist()
}
assert len(episodes) == info["total_episodes"]
print(f"Dataset: {DATASET_ROOT}")
print(f"Episodes: {len(episodes)} | Frames: {info['total_frames']} | FPS: {info['fps']}")
print("Tasks:")
for task in sorted({task for episode in episodes.values() for task in episode["tasks"]}):
    count = sum(task in episode["tasks"] for episode in episodes.values())
    print(f"  {task}: {count}")


## 选择一条轨迹

`EPISODE_INDEX` 范围为 `0` 到 `153`。第一次运行会用 ffmpeg 从原始 AV1 视频抽取这条轨迹并转成浏览器可播放的 H.264；缓存保存在数据集的 `visualizations/` 中。视频画面左边是 FPV，右边是俯视相机。曲线中的 state 是本帧观测到的关节位置，action 是数据集记录的目标动作；数值单位沿用数据集原始字段。两条线不要求在同一帧完全重合：观测通常先于该帧动作，而接触物体或动作限幅也可能带来持续差异。数据集的 action 不保证等于电机实际收到的最终指令。


In [ ]:
EPISODE_INDEX = int(os.environ.get("ORANGE_CUBE_EPISODE_INDEX", "0"))


def video_path(episode, key):
    chunk = episode[f"videos/{key}/chunk_index"]
    file_index = episode[f"videos/{key}/file_index"]
    return DATASET_ROOT / info["video_path"].format(
        video_key=key, chunk_index=chunk, file_index=file_index
    )


def make_episode_video(episode):
    output_dir = DATASET_ROOT / "visualizations"
    output_dir.mkdir(exist_ok=True)
    output = output_dir / f"episode_{episode['episode_index']:03d}_dual_camera.mp4"
    if output.is_file() and output.stat().st_size > 0:
        return output
    fpv_key = "observation.images.fpv"
    top_key = "observation.images.top"
    fpv = video_path(episode, fpv_key)
    top = video_path(episode, top_key)
    assert fpv.is_file() and top.is_file(), f"Missing source video: {fpv} or {top}"
    duration = episode["length"] / info["fps"]
    filter_graph = (
        "[0:v]scale=640:360:force_original_aspect_ratio=decrease,"
        "pad=640:360:(ow-iw)/2:(oh-ih)/2[fpv];"
        "[1:v]scale=640:360:force_original_aspect_ratio=decrease,"
        "pad=640:360:(ow-iw)/2:(oh-ih)/2[top];"
        "[fpv][top]hstack=inputs=2[out]"
    )
    command = [
        "ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
        "-ss", str(episode[f"videos/{fpv_key}/from_timestamp"]), "-i", str(fpv),
        "-ss", str(episode[f"videos/{top_key}/from_timestamp"]), "-i", str(top),
        "-filter_complex", filter_graph, "-map", "[out]", "-t", str(duration),
        "-an", "-c:v", "libx264", "-preset", "veryfast", "-crf", "25",
        "-pix_fmt", "yuv420p", "-movflags", "+faststart", str(output),
    ]
    subprocess.run(command, check=True)
    return output


def episode_frame(episode):
    data_path = DATASET_ROOT / info["data_path"].format(
        chunk_index=episode["data/chunk_index"], file_index=episode["data/file_index"]
    )
    table = pq.read_table(
        data_path,
        columns=["episode_index", "timestamp", "frame_index", "action", "observation.state"],
        filters=[("episode_index", "=", episode["episode_index"])],
    )
    assert table.num_rows == episode["length"], (
        f"Expected {episode['length']} frames, found {table.num_rows}"
    )
    timestamps = table["timestamp"].to_numpy()
    state = np.asarray(table["observation.state"].to_pylist(), dtype=float)
    action = np.asarray(table["action"].to_pylist(), dtype=float)
    return timestamps, state, action


def show_episode(index):
    episode = episodes[int(index)]
    tasks = ", ".join(episode["tasks"])
    duration = episode["length"] / info["fps"]
    display(Markdown(f"### Episode {index}: {tasks}  \n{episode['length']} frames · {duration:.1f} s"))
    timestamps, state, action = episode_frame(episode)
    names = info["features"]["action"]["names"]
    fig, axes = plt.subplots(3, 2, figsize=(14, 9), sharex=True)
    for joint, ax in enumerate(axes.flat):
        ax.plot(timestamps, state[:, joint], label="state", linewidth=1.3)
        ax.plot(timestamps, action[:, joint], label="action", linewidth=1.0, alpha=0.8)
        ax.set_ylabel(names[joint])
        ax.grid(alpha=0.25)
    axes.flat[0].legend()
    for ax in axes[-1]:
        ax.set_xlabel("Time in episode (s)")
    fig.suptitle(f"SO-101 joint trajectories · episode {index}")
    fig.tight_layout()
    preview_dir = DATASET_ROOT / "visualizations"
    preview_dir.mkdir(exist_ok=True)
    chart = preview_dir / f"episode_{index:03d}_joint_curves.png"
    fig.savefig(chart, dpi=150)
    display(fig)
    plt.close(fig)
    clip = make_episode_video(episode)
    poster = preview_dir / f"episode_{index:03d}_preview.jpg"
    if not poster.is_file():
        subprocess.run([
            "ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
            "-ss", str(duration / 2), "-i", str(clip), "-frames:v", "1", str(poster),
        ], check=True)
    display(Video(filename=str(clip), embed=True, width=960))
    print(f"Cached clip: {clip}")
    print(f"Preview: {poster} | Chart: {chart}")


show_episode(EPISODE_INDEX)


## 单帧样本的实际类型

读取当前 episode 的第 `SAMPLE_FRAME_INDEX` 帧，把 `LeRobotDataset` 返回的字段整理成字典，并按字段展示类型。Tensor 只显示 `shape` 和 `dtype`，不会打印整张图像的数据。


In [ ]:
SAMPLE_FRAME_INDEX = 0
assert 0 <= SAMPLE_FRAME_INDEX < episodes[EPISODE_INDEX]["length"], (
    "SAMPLE_FRAME_INDEX is outside the selected episode"
)
SAMPLE_INDEX = episodes[EPISODE_INDEX]["dataset_from_index"] + SAMPLE_FRAME_INDEX
LEROBOT_PYTHON = os.environ.get("LEROBOT_PYTHON", "/opt/venvs/carrot/bin/python")

inspect_code = """
import json
import sys

import torch
from lerobot.datasets import LeRobotDataset

dataset = LeRobotDataset("felixmayor/orange_cube_merged", root=sys.argv[1])
sample = dataset[int(sys.argv[2])]
summary = {}
for key, value in sample.items():
    entry = {"type": f"{type(value).__module__}.{type(value).__qualname__}"}
    if isinstance(value, torch.Tensor):
        entry["shape"] = list(value.shape)
        entry["dtype"] = str(value.dtype)
    summary[key] = entry
print(json.dumps(summary, ensure_ascii=False, indent=4))
"""
result = subprocess.run(
    [LEROBOT_PYTHON, "-c", inspect_code, str(DATASET_ROOT), str(SAMPLE_INDEX)],
    check=True,
    capture_output=True,
    text=True,
    env={**os.environ, "HF_HUB_OFFLINE": "1"},
)
sample_types = json.loads(result.stdout)
rows = ["| field | type | shape | dtype |", "| --- | --- | --- | --- |"]
for field, details in sample_types.items():
    shape = str(details["shape"]) if "shape" in details else "—"
    dtype = details.get("dtype", "—")
    rows.append(f"| `{field}` | `{details['type']}` | `{shape}` | `{dtype}` |")
display(Markdown(f"**dataset[{SAMPLE_INDEX}]** · episode {EPISODE_INDEX} · frame {SAMPLE_FRAME_INDEX}\n\n" + "\n".join(rows)))


## 交互浏览

下拉框可切换全部 154 条轨迹；首次选择新轨迹时需要等待视频转码。静态导出的 notebook 仍包含上面那条已执行的示例。


In [ ]:
selector = widgets.Dropdown(
    options=[
        (f"{index:03d} · {', '.join(episode['tasks'])}", index)
        for index, episode in sorted(episodes.items())
    ],
    value=EPISODE_INDEX,
    description="Episode",
    layout=widgets.Layout(width="540px"),
)
interactive = widgets.Output()


def on_episode_change(change):
    if change["name"] == "value":
        with interactive:
            interactive.clear_output(wait=True)
            show_episode(change["new"])


selector.observe(on_episode_change, names="value")
display(selector, interactive)
